This is a file for prototyping using the models/dataloaders defined elsewhere. In addition, it contains the training loop but should NOT include the actual models themselves

In [ ]:
import os
import sys

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader, Dataset

from transformers import get_constant_schedule_with_warmup
from tqdm import tqdm

torch.manual_seed(8008135)

# drive.mount('/content/drive', force_remount=True)

base_dir = ".."

assert os.path.exists(base_dir), f"Path does not exist: {base_dir}"
print("base_dir contents:", os.listdir(base_dir))

if base_dir not in sys.path:
    sys.path.insert(0, base_dir)

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)
print(f"Device set to {device}")

base_dir contents: ['Datasets', '.DS_Store', 'Model']
Device set to cuda


In [7]:
from Model.HRM_Model import HRM
from Model.HRM_Components import Encoder, HighLevel, LowLevel, Head
from Datasets.Sudoku_DataLoader import get_loaders

In [19]:
train_size = 2**17
test_size = 2**15
batch_size = 128
train_dataloader, val_dataloader = get_loaders(train_size, test_size, batch_size)

Map: 100%|██████████| 32768/32768 [00:00<00:00, 41489.46 examples/s]


In [ ]:
#Define Model Hyperparameters
# d_model = 512
# M = 4
# N = 2
# T = 4
# n_layers = 4
# n_heads = 8
d_model = 257
M = 3
N = 2
T = 2
n_layers = 2
n_heads = 4
vocab_size = 10
lr = 1e-4
beta1 = .9
beta2 = .95
weight_decay = .1
num_epochs = 100
num_warmup_steps = 200
dropout = 0.2

high_level = HighLevel(d_model=d_model, n_layers=n_layers, n_heads=n_heads, intermediate_size=4*d_model, dropout=dropout)
low_level = LowLevel(d_model=d_model, n_layers=n_layers, n_heads=n_heads, intermediate_size=4*d_model, dropout=dropout)
head = Head(vocab_size=vocab_size, d_model=d_model)
encoder = Encoder(vocab_size=vocab_size, d_model=d_model)


HRM_model = HRM(low_level, high_level, encoder, head, M, N, T).to(device)

# This optimizer isn't technically paper-accurate, but we found AdamW to
# work effectively
optimizer = optim.AdamW(
    HRM_model.parameters(),
    lr=lr,
    betas=(beta1, beta2),
    weight_decay=weight_decay
)

num_training_steps = len(train_dataloader) * num_epochs
num_warmup_steps = num_warmup_steps

scheduler = get_constant_schedule_with_warmup(
    optimizer,
    num_warmup_steps=num_warmup_steps,
)

In [22]:
def save_checkpoint(model, optimizer, scheduler, epoch, path, best_board_acc=None):
    os.makedirs(os.path.dirname(path), exist_ok=True)

    checkpoint = {
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
    }

    if scheduler is not None:
        checkpoint["scheduler_state_dict"] = scheduler.state_dict()

    if best_board_acc is not None:
        checkpoint["best_board_acc"] = best_board_acc

    torch.save(checkpoint, path)


def load_checkpoint(model, optimizer, scheduler, path, device):
    checkpoint = torch.load(path, map_location=device)
    model.load_state_dict(checkpoint["model_state_dict"])
    optimizer.load_state_dict(checkpoint["optimizer_state_dict"])

    if scheduler is not None and "scheduler_state_dict" in checkpoint:
        scheduler.load_state_dict(checkpoint["scheduler_state_dict"])

    start_epoch = checkpoint["epoch"] + 1
    return model, optimizer, scheduler, start_epoch

In [23]:
@torch.no_grad()
def evaluate_hrm(
    model,
    val_loader,
    loss_fn,
    device,
):
    model.eval()

    total_loss = 0.0
    correct_tokens = 0
    total_tokens = 0
    correct_boards = 0
    total_boards = 0

    for x, y in tqdm(val_loader, desc="Validation", leave=False):
        x = x.to(device)
        y = y.to(device)

        z_H = None
        z_L = None
        logits = None

        for _ in range(model.M):
            z_H, z_L, logits = model.segment(x, z_H, z_L)
            z_H = z_H.detach()
            z_L = z_L.detach()

        x_flat = x.reshape(-1)
        y_flat = y.reshape(-1)

        mask = (x_flat != y_flat)
        targets = y_flat.clone()
        targets[~mask] = -100

        pred = logits.reshape(-1, logits.size(-1))
        loss = loss_fn(pred, targets)
        total_loss += loss.item()

        pred_labels = pred.argmax(dim=1)

        # Token accuracy on unknown cells only
        correct_tokens += (pred_labels[mask] == y_flat[mask]).sum().item()
        total_tokens += mask.sum().item()

        # Board accuracy
        pred_board = pred_labels.view_as(y)
        filled_board = torch.where(x != 0, x, pred_board)
        board_correct = (filled_board == y).all(dim=1)

        correct_boards += board_correct.sum().item()
        total_boards += y.size(0)

    avg_loss = total_loss / len(val_loader)
    token_acc = correct_tokens / total_tokens if total_tokens > 0 else 0.0
    board_acc = correct_boards / total_boards if total_boards > 0 else 0.0

    return avg_loss, token_acc, board_acc

In [24]:
def train_hrm_deepsup(
    model,
    train_loader,
    optimizer,
    loss_fn,
    device,
    scheduler=None,
    num_epochs=10,
    checkpoint_dir="checkpoints",
    checkpoint_every=5,
    val_loader=None,
):
    print(f"Number of trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

    best_metric = 0.0

    for epoch in range(num_epochs):
        model.train()

        epoch_loss = 0.0
        correct_tokens = 0
        total_tokens = 0
        correct_boards = 0
        total_boards = 0

        for x, y in tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}"):
            x = x.to(device)
            y = y.to(device)

            z_H = None
            z_L = None
            batch_loss = 0.0
            last_pred = None

            x_flat = x.reshape(-1)
            y_flat = y.reshape(-1)

            # Only supervise unknown Sudoku cells
            mask = (x_flat != y_flat)
            targets = y_flat.clone()
            targets[~mask] = -100

            for _ in range(model.M):
                optimizer.zero_grad(set_to_none=True)

                z_H, z_L, logits = model.segment(x, z_H, z_L)

                pred = logits.reshape(-1, logits.size(-1))
                loss = loss_fn(pred, targets)

                loss.backward()
                optimizer.step()

                if scheduler is not None:
                    scheduler.step()

                z_H = z_H.detach()
                z_L = z_L.detach()

                batch_loss += loss.item()
                last_pred = pred.detach()

            epoch_loss += batch_loss / model.M

            with torch.no_grad():
                pred_labels = last_pred.argmax(dim=1)

                # Token accuracy on unknown cells only
                correct_tokens += (pred_labels[mask] == y_flat[mask]).sum().item()
                total_tokens += mask.sum().item()

                # Board accuracy
                pred_board = pred_labels.view_as(y)
                filled_board = torch.where(x != 0, x, pred_board)
                board_correct = (filled_board == y).all(dim=1)

                correct_boards += board_correct.sum().item()
                total_boards += y.size(0)

        avg_loss = epoch_loss / len(train_loader)
        token_acc = correct_tokens / total_tokens if total_tokens > 0 else 0.0
        board_acc = correct_boards / total_boards if total_boards > 0 else 0.0
        current_lr = optimizer.param_groups[0]["lr"]

        print(
            f"Epoch {epoch+1}: "
            f"Avg Train Loss = {avg_loss:.4f}, "
            f"Train Token Accuracy = {token_acc*100:.2f}%, "
            f"Train Board Accuracy = {board_acc*100:.2f}%, "
            f"LR = {current_lr:.2e}"
        )

        val_loss = None
        val_token_acc = None
        val_board_acc = None

        if val_loader is not None:
            val_loss, val_token_acc, val_board_acc = evaluate_hrm(
                model,
                val_loader,
                loss_fn,
                device,
            )

            print(
                f"Val Loss = {val_loss:.4f}, "
                f"Val Token Accuracy = {val_token_acc*100:.2f}%, "
                f"Val Board Accuracy = {val_board_acc*100:.2f}%"
            )

        # Use validation board accuracy for model selection if available
        metric_for_best = val_board_acc if val_board_acc is not None else board_acc

        # Save latest every epoch
        save_checkpoint(
            model,
            optimizer,
            scheduler,
            epoch,
            os.path.join(checkpoint_dir, "hrm_last.pt"),
            best_board_acc=best_metric,
        )

        # Save best model
        if metric_for_best > best_metric:
            best_metric = metric_for_best
            save_checkpoint(
                model,
                optimizer,
                scheduler,
                epoch,
                os.path.join(checkpoint_dir, "hrm_best.pt"),
                best_board_acc=best_metric,
            )

        # Save periodic checkpoint
        if checkpoint_every is not None and (epoch + 1) % checkpoint_every == 0:
            save_checkpoint(
                model,
                optimizer,
                scheduler,
                epoch,
                os.path.join(checkpoint_dir, f"hrm_epoch_{epoch+1}.pt"),
                best_board_acc=best_metric,
            )

    return model, best_metric

In [25]:
HRM_model, best_metric = train_hrm_deepsup(
    HRM_model,
    train_dataloader,
    optimizer,
    nn.CrossEntropyLoss(ignore_index=-100),
    device=device,
    scheduler=scheduler,
    num_epochs=num_epochs,
    checkpoint_dir="checkpoints",
    checkpoint_every=5,
    val_loader=val_dataloader,
)

HRM_model.eval()
print("Best board accuracy used for checkpointing:", best_metric)

Number of trainable parameters: 4,201,472


Epoch 1/100: 100%|██████████| 1024/1024 [02:02<00:00,  8.34it/s]


Epoch 1: Avg Train Loss = 1.5520, Train Token Accuracy = 32.60%, Train Board Accuracy = 0.00%, LR = 1.00e-04


Val Loss = 1.2119, Val Token Accuracy = 45.02%, Val Board Accuracy = 0.01%


Epoch 2/100: 100%|██████████| 1024/1024 [02:02<00:00,  8.36it/s]


...

Epoch 99: Avg Train Loss = 0.7224, Train Token Accuracy = 77.15%, Train Board Accuracy = 36.34%, LR = 1.00e-04


Val Loss = 0.6147, Val Token Accuracy = 75.20%, Val Board Accuracy = 41.32%


Epoch 100/100: 100%|██████████| 1024/1024 [02:01<00:00,  8.44it/s]


Epoch 100: Avg Train Loss = 0.7223, Train Token Accuracy = 77.12%, Train Board Accuracy = 36.30%, LR = 1.00e-04


Val Loss = 0.5960, Val Token Accuracy = 75.51%, Val Board Accuracy = 41.70%
Best board accuracy used for checkpointing: 0.4169921875


In [32]:
def print_sudoku_comparison(x, pred, y):
    x = x.view(9, 9)
    pred = pred.view(9, 9)
    y = y.view(9, 9)

    RED = "\033[91m"
    RESET = "\033[0m"

    print("Input / Prediction (errors marked)")

    for i in range(9):
        if i % 3 == 0 and i != 0:
            print("-" * 33)

        row = ""
        for j in range(9):
            if j % 3 == 0 and j != 0:
                row += "| "

            if x[i, j] != 0:
                # given clue
                val = str(x[i, j].item())
            else:
                if pred[i, j] == y[i, j]:
                    val = str(pred[i, j].item())
                else:
                    # color incorrect prediction red
                    val = f"{RED}{pred[i, j].item()}{RESET}"

            row += val + " "

        print(row)
    print()

In [33]:
#Visualize a few sudokus
data_iter = iter(val_dataloader)
x_batch, y_batch = next(data_iter)

for i in range(10):
    x = x_batch[i]
    y = y_batch[i]

    x = torch.tensor(x, dtype=torch.long)

    pred = HRM_model.predict(x).cpu()

    print_sudoku_comparison(x, pred, y)

    cor_tok = (pred == y).sum()
    print("Token Accuracy:", cor_tok.item() / 81)

Input / Prediction (errors marked)
5 9 8 | 4 6 1 | 2 7 3 
7 3 1 | 5 2 8 | 4 9 6 
2 6 4 | 3 9 7 | 5 8 1 
---------------------------------
8 7 9 | 6 3 2 | 1 4 5 
4 2 6 | 1 5 9 | 7 3 8 
1 5 3 | 8 7 4 | 9 6 2 
---------------------------------
6 1 5 | 9 4 3 | 8 2 7 
3 4 2 | 7 8 5 | 6 1 9 
9 8 7 | 2 1 6 | 3 5 4 

Token Accuracy: 1.0
Input / Prediction (errors marked)
2 9 7 | 4 5 6 | 1 8 3 
3 5 2 | 2 8 1 | 9 6 7 
8 6 1 | 7 9 3 | 2 4 5 
---------------------------------
9 3 4 | 6 2 5 | 7 1 8 
5 1 2 | 8 7 4 | 6 3 9 
8 7 6 | 1 3 9 | 4 5 2 
---------------------------------
7 8 9 | 5 1 2 | 3 6 4 
1 4 5 | 3 6 7 | 8 2 9 
6 2 3 | 9 4 8 | 5 7 1 

Token Accuracy: 0.8518518518518519
Input / Prediction (errors marked)
5 2 9 | 4 7 8 | 3 6 1 
8 6 7 | 9 1 3 | 2 4 5 
1 4 3 | 5 2 6 | 9 8 7 
---------------------------------
4 3 5 | 6 8 1 | 7 2 9 
2 9 2 | 3 4 5 | 8 1 6 
6 1 6 | 7 9 2 | 4 5 3 
---------------------------------
9 8 4 | 1 3 6 | 5 7 2 
6 7 4 | 8 5 9 | 1 3 2 
3 5 1 | 2 8 4 | 6 9 8 

Token Accura

/tmp/ipykernel_17192/1568551438.py:9: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  x = torch.tensor(x, dtype=torch.long)


In [14]:
#Save the model
torch.save(HRM_model.state_dict(), "hrm_model.pt")